# 03 — الـ Pipeline كامل End-to-End (`test_pipeline`)

بنمشي رحلة المستخدم بالكامل على الـ API الحقيقي + Postgres/pgvector الحقيقي:

`register → workspace → project → upload (txt / عربي / .py / pdf) → ingestion → chunks+embeddings في الـ DB → search (EN / AR / cross-lingual) → scoped search → isolation → chat + citations → re-upload (versioning) → delete → failure paths`

كل مرحلة بتتقاس بالزمن، وفي الآخر فيه جداول ورسومات. الـ LLM هنا **fake** عشان النتايج تبقى ثابتة وبدون تكلفة.

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd()))
sys.path.insert(0, str(pathlib.Path.cwd() / "notebooks"))
from nb_utils import *
env = setup()
print(env)
rec = Recorder("03_pipeline_e2e")

In [ ]:
import matplotlib.pyplot as plt
from sqlalchemy import text, inspect as sa_inspect

api = get_api()
corpus = make_corpus()
display(pd.DataFrame([(k, m, p or "— NOT FOUND —") for k, (m, p) in api.routes.items()], columns=["key", "method", "path"]))
A = B = WS = PID = None
FILES = {}

## 1) المستخدمين والـ Hierarchy

In [ ]:
with rec.case("register + login two users (alice, bob)", "auth") as c:
    A = api.new_user("alice"); B = api.new_user("bob")
    c.note(A.email, B.email)

with rec.case("create workspace and project", "hierarchy") as c:
    need(A)
    WS = api.make_workspace(A); PID = api.make_project(A, WS)
    c.note(f"workspace={WS}", f"project={PID}")

with rec.case("workspace can be read back by its owner", "hierarchy"):
    need(A, WS)
    r = api.route("workspace_get", WS, token=A.token); assert r.status_code == 200, r.text[:150]

## 2) Upload + Ingestion
بنرفع كل الملفات وبنستنى لحد ما الـ status يبقى READY (وبنقيس الزمن).

In [ ]:
for name, (data, mime) in corpus.items():
    with rec.case(f"upload {name}", "upload") as c:
        need(A, PID)
        t0 = time.perf_counter(); r = api.upload(A, PID, name, data, mime); up = time.perf_counter() - t0
        assert r.status_code in (200, 201, 202), f"HTTP {r.status_code}: {r.text[:200]}"
        fid = get_id(r); assert fid, "no id in upload response"
        FILES[name] = dict(id=fid, size_kb=round(len(data) / 1024, 1), upload_s=round(up, 2))
        c.note(f"id={fid}", f"HTTP {r.status_code}", f"{up:.2f}s")
if "operations_report.pdf" not in corpus:
    rec.skip("upload operations_report.pdf", "no PDF library (pymupdf/reportlab) to generate the sample PDF", "upload")

for name, info in FILES.items():
    with rec.case(f"ingestion reaches READY: {name}", "ingestion") as c:
        status, secs = api.wait_ready(A, info["id"])
        info.update(status=status, ingest_s=round(secs, 1))
        assert status in CONFIG["READY_STATUSES"], f"final status={status} after {secs:.1f}s"
        c.note(f"{secs:.1f}s")

## 3) تحقق مباشر من الداتابيز
مش كفاية الـ API يقول READY — بنتأكد إن الـ chunks والـ embeddings موجودة فعلاً بالـ dimension الصح.

In [ ]:
for name, info in FILES.items():
    with rec.case(f"chunks + embeddings stored: {name}", "database") as c:
        with get_session() as s:
            st = chunk_stats(s, info["id"])
        info.update(chunks=st["total"])
        assert st["total"] > 0, "no chunks stored for this file"
        assert st["embedded"] == st["total"], f"{st['total'] - st['embedded']} chunks have NO embedding"
        assert st["min_dim"] == st["max_dim"] == CONFIG["EMBED_DIM"], f"dims {st['min_dim']}..{st['max_dim']}"
        c.note(f"{st['total']} chunks, dim={st['max_dim']}")

with rec.case("chunk text quality: no empty / duplicate chunks", "database") as c:
    need(FILES)
    with get_session() as s:
        t, v = chunk_table(s); tc = text_column(s, t); fk = CONFIG["CHUNK_FILE_FK"]
        if not tc:
            raise Skip("could not find the chunk text column (content/text/chunk_text)")
        ids = [str(i["id"]) for i in FILES.values()]
        n, empty, distinct, avg, mx = s.execute(text(
            f'select count(*), coalesce(sum(case when length(trim("{tc}"))=0 then 1 else 0 end),0), count(distinct md5("{tc}")), '
            f'avg(length("{tc}")), max(length("{tc}")) from "{t}" where "{fk}"::text = any(:ids)'), {"ids": ids}).one()
    c.note(f"{n} chunks, avg {float(avg):.0f} chars, max {mx}")
    assert empty == 0, f"{empty} empty chunks"
    assert distinct == n, f"{n - distinct} duplicated chunks"

with rec.case("PDF chunks keep page numbers (page-2 fact has a higher page than page-1 text)", "database") as c:
    need(FILES.get("operations_report.pdf"))
    with get_session() as s:
        t, v = chunk_table(s); tc = text_column(s, t)
        cols = [x["name"] for x in sa_inspect(s.get_bind()).get_columns(t)]
        pc = next((x for x in ("page_number", "page", "page_no") if x in cols), None)
        if not pc or not tc:
            raise Skip("no page column / text column on the chunk table")
        q = f'select "{pc}" from "{t}" where "{CONFIG["CHUNK_FILE_FK"]}"::text=:f and "{tc}" ilike :k order by 1 limit 1'
        fid = str(FILES["operations_report.pdf"]["id"])
        p1 = s.execute(text(q), {"f": fid, "k": "%Regional Operations%"}).scalar()
        p2 = s.execute(text(q), {"f": fid, "k": "%3.7 percent%"}).scalar()
    assert p1 is not None and p2 is not None, f"page values missing (intro page={p1}, fact page={p2})"
    c.note(f"intro page={p1}, fact page={p2}")
    assert int(p2) > int(p1), "both chunks report the same page -> page numbers are not tracked per page"

## 4) Retrieval — جودة البحث
Golden set: 10 أسئلة (إنجليزي، عربي، cross-lingual، كود، PDF). بنقيس **hit@1 / hit@3 / MRR**.

In [ ]:
ret_rows = []
for q, lang, kw, fname in QA:
    with rec.case(f"retrieve [{lang}] {q[:48]}", "retrieval") as c:
        need(A, PID)
        if fname not in FILES:
            raise Skip(f"{fname} was not uploaded")
        t0 = time.perf_counter(); r = api.search(A, PID, q); ms = (time.perf_counter() - t0) * 1000
        assert r.status_code == 200, f"HTTP {r.status_code}: {r.text[:200]}"
        hits = find_hits(safe_json(r)); assert hits, "no results returned"
        rk = rank_of(hits, kw)
        ret_rows.append(dict(lang=lang, question=q[:50], expected=kw, rank=rk, ms=round(ms), top1=hit_blob(hits[0])[:70]))
        assert rk is not None, f"'{kw}' not found in the top-{len(hits)} results"
        c.note(f"rank={rk}", f"{ms:.0f}ms")

if ret_rows:
    R = pd.DataFrame(ret_rows)
    R["hit1"] = R["rank"].eq(1); R["hit3"] = R["rank"].between(1, 3); R["rr"] = R["rank"].map(lambda x: 1 / x if pd.notna(x) else 0)
    by = R.groupby("lang")[["hit1", "hit3", "rr"]].mean().rename(columns={"rr": "MRR"}).round(2)
    display(R); display(by)
    tot = dict(hit1=R.hit1.mean(), hit3=R.hit3.mean(), MRR=R.rr.mean())
    print({k: round(v, 3) for k, v in tot.items()})
    rec.check(f"overall hit@3 >= {CONFIG['THRESHOLDS']['hit_at_3']}", tot["hit3"] >= CONFIG["THRESHOLDS"]["hit_at_3"], f"hit@3={tot['hit3']:.2f}", "retrieval")
    by[["hit1", "hit3", "MRR"]].plot.bar(figsize=(8, 3.2), title="Retrieval quality by query type", ylim=(0, 1.05)); plt.show()

In [ ]:
with rec.case("file-scoped search only returns chunks of the chosen file", "retrieval") as c:
    need(A, PID, FILES.get("orion_manual.txt"))
    oid = FILES["orion_manual.txt"]["id"]
    r = api.search(A, PID, "ما هي ميزانية مشروع النيل الأزرق؟", file_ids=[oid])
    assert r.status_code == 200, f"HTTP {r.status_code}: {r.text[:200]}"
    hits = find_hits(safe_json(r))
    if not hits:
        c.note("0 hits (acceptable: nothing relevant inside the chosen file)")
    else:
        if not any(str(oid).lower() in hit_blob(h) or "orion_manual" in hit_blob(h) for h in hits):
            raise Skip("search results do not expose file id/name, cannot verify scoping")
        leaked = [h for h in hits if "blue_nile" in hit_blob(h) or "النيل الأزرق" in hit_blob(h)]
        assert not leaked, f"{len(leaked)} result(s) from another file leaked into a file-scoped search"

with rec.case("multi-file subset search (2 of 4 files)", "retrieval") as c:
    need(A, PID); ids = [FILES[n]["id"] for n in ("orion_manual.txt", "inventory.py") if n in FILES]
    if len(ids) < 2: raise Skip("needs both orion_manual.txt and inventory.py")
    r = api.search(A, PID, "What temperature does the reactor operate at?", file_ids=ids)
    assert r.status_code == 200; hits = find_hits(safe_json(r)); assert hits
    assert rank_of(hits, "450") is not None, "expected fact missing from the subset search"

## 5) Isolation بين المستخدمين

In [ ]:
with rec.case("another user cannot search Alice's project", "isolation") as c:
    need(A, B, PID)
    r = api.search(B, PID, "Orion-7 reactor temperature")
    hits = find_hits(safe_json(r)) if r.status_code == 200 else []
    assert r.status_code in (401, 403, 404) or not hits, f"HTTP {r.status_code} with {len(hits)} hit(s) -> DATA LEAK"
    c.note(f"HTTP {r.status_code}")
with rec.case("another user cannot read Alice's file", "isolation") as c:
    need(B, FILES.get("orion_manual.txt"))
    r = api.route("file_get", FILES["orion_manual.txt"]["id"], token=B.token)
    assert r.status_code in (403, 404), f"HTTP {r.status_code} -> file visible to a non-member"

## 6) Chat + Citations

In [ ]:
CONV = None
with rec.case("create conversation", "chat") as c:
    need(A, PID); CONV = api.make_conversation(A, PID); c.note(f"conversation={CONV}")

CHAT = {}
with rec.case("send message -> assistant answer (fake LLM)", "chat") as c:
    need(A, CONV)
    t0 = time.perf_counter(); r = api.send_message(A, CONV, "What temperature does the Orion-7 reactor operate at?"); dt = time.perf_counter() - t0
    assert r.status_code in (200, 201), f"HTTP {r.status_code}: {r.text[:250]}"
    js = unwrap(safe_json(r)); CHAT["js"] = js
    am = js.get("assistant_message", js) if isinstance(js, dict) else js
    body = " ".join(flatten_strings(am.get("blocks", am) if isinstance(am, dict) else am))
    CHAT["text"], CHAT["citations"] = body, (find_key(am, "citations") or [])
    assert body.strip(), "assistant message is empty"
    c.note(f"{dt:.2f}s", f"{len(CHAT['citations'])} citation(s)")

with rec.case("answer carries citations", "chat"):
    need(CHAT.get("js")); assert CHAT["citations"], "no citations returned (Explainable RAG requirement)"
with rec.case("every [n] marker maps to citations[n-1]", "chat") as c:
    need(CHAT.get("js"))
    marks = {int(x) for x in re.findall(r"\[(\d+)\]", CHAT["text"])}
    if not marks: raise Warn("no [n] markers in the answer text")
    assert max(marks) <= len(CHAT["citations"]), f"marker [{max(marks)}] but only {len(CHAT['citations'])} citations"
with rec.case("cited chunk_ids really exist in the DB", "chat") as c:
    need(CHAT.get("js"))
    ids = [str(x.get("chunk_id")) for x in CHAT["citations"] if isinstance(x, dict) and x.get("chunk_id")]
    if not ids: raise Skip("citations have no chunk_id field")
    with get_session() as s:
        t, _ = chunk_table(s)
        n = s.execute(text(f'select count(*) from "{t}" where id::text = any(:ids)'), {"ids": ids}).scalar()
    assert n == len(set(ids)), f"{len(set(ids)) - n} cited chunk(s) do not exist"

## 7) Versioning (إعادة رفع نفس الملف) والحذف

In [ ]:
with rec.case("re-upload changed file -> new content is searchable, old one is not preferred", "versioning") as c:
    need(A, PID, FILES.get("orion_manual.txt"))
    new = corpus["orion_manual.txt"][0].decode().replace("450 degrees", "500 degrees").encode()
    r = api.upload(A, PID, "orion_manual.txt", new, "text/plain")
    assert r.status_code in (200, 201, 202), f"HTTP {r.status_code}: {r.text[:200]}"
    fid2 = get_id(r); st, secs = api.wait_ready(A, fid2); assert st in CONFIG["READY_STATUSES"], st
    hits = find_hits(safe_json(api.search(A, PID, "What temperature does the Orion-7 reactor operate at?")))
    r_new, r_old = rank_of(hits, "500 degrees"), rank_of(hits, "450 degrees")
    c.note(f"new-version rank={r_new}, old-version rank={r_old}, same file id reused={str(fid2) == str(FILES['orion_manual.txt']['id'])}")
    assert r_new is not None, "new version is not searchable"
    if r_old is not None and r_old < r_new:
        raise Warn("OLD version ranks above the new one - decide if old versions must be excluded from search")

with rec.case("delete file -> chunks removed, file gone, no longer searchable", "delete") as c:
    need(A, PID, FILES.get("inventory.py"))
    fid = FILES["inventory.py"]["id"]
    r = api.route("file_delete", fid, token=A.token); assert r.status_code in (200, 202, 204), f"HTTP {r.status_code}: {r.text[:150]}"
    with get_session() as s: left = chunk_stats(s, fid)["total"]
    assert left == 0, f"{left} orphan chunks left after delete"
    assert api.route("file_get", fid, token=A.token).status_code == 404, "deleted file still readable"
    hits = find_hits(safe_json(api.search(A, PID, "Which function calculates the reorder point?")))
    assert rank_of(hits, "calculate_reorder_point") is None, "deleted file still returned by search"

## 8) Failure paths — لازم يرجع 4xx / FAILED مش 500

In [ ]:
def upload_case(name, data, mime):
    need(A, PID)
    r = api.upload(A, PID, name, data, mime)
    assert r.status_code < 500, f"HTTP {r.status_code} (server error): {r.text[:200]}"
    info = f"HTTP {r.status_code}"
    if r.status_code < 300:
        fid = get_id(r)
        st, _ = api.wait_ready(A, fid, timeout=60)
        info += f", final status={st}"
    return r, info

with rec.case("unsupported file type (.exe) is rejected cleanly", "failure-paths") as c:
    r, info = upload_case("malware.exe", b"MZ\x90\x00\x03", "application/x-msdownload"); c.note(info)
with rec.case("empty .txt file does not crash ingestion", "failure-paths") as c:
    r, info = upload_case("empty.txt", b"", "text/plain"); c.note(info)
with rec.case("corrupted PDF ends FAILED/4xx (not a stuck 'processing')", "failure-paths") as c:
    r, info = upload_case("broken.pdf", b"%PDF-1.4 not really a pdf", "application/pdf"); c.note(info)
    if "final status=" in info and any(x in info for x in ("PROCESSING", "PENDING", "UPLOADED", "TIMEOUT")):
        raise AssertionError("file stuck in a non-final status: " + info)
with rec.case("path-traversal filename '../../evil.txt' is sanitised", "failure-paths") as c:
    r, info = upload_case("../../evil.txt", b"hello", "text/plain")
    if r.status_code < 300:
        assert "../" not in " ".join(flatten_strings(safe_json(r))), "response still contains '../' in a stored name/path"
    c.note(info)
with rec.case("very long filename (400 chars)", "failure-paths") as c:
    r, info = upload_case("a" * 400 + ".txt", b"hello", "text/plain"); c.note(info)

## 9) جدول الزمن والرسومات

In [ ]:
if FILES:
    F = pd.DataFrame.from_dict(FILES, orient="index").drop(columns=["id"], errors="ignore")
    display(F)
    fig, ax = plt.subplots(1, 2, figsize=(11, 3.4))
    F[["upload_s", "ingest_s"]].plot.bar(ax=ax[0], title="Upload vs ingestion time (s)"); ax[0].tick_params(axis="x", rotation=25)
    if "chunks" in F: F["chunks"].plot.bar(ax=ax[1], color="#4c78a8", title="Chunks per file"); ax[1].tick_params(axis="x", rotation=25)
    plt.tight_layout(); plt.show()
L = pd.DataFrame(api.log)
if len(L):
    display(L.groupby(["method", "status"]).ms.agg(["count", "mean", "max"]).round(1))
    rec.check("no HTTP 5xx during the whole pipeline run", not (L.status >= 500).any(), f"{int((L.status >= 500).sum())} server errors", "http")

In [ ]:
rec.summary(); rec.table(); rec.save()